# ДЗ №1. Задание 3
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Сходство фильмов.


In [31]:
import csv
import os
import sys
from math import sqrt
from pyspark import SparkConf, SparkContext

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable

conf = SparkConf().setMaster("local[2]").setAppName("HW1-Movies")
sc = SparkContext.getOrCreate(conf=conf)


## Чтение данных

Для каждого фильма собираем словарь `{userId: rating}`. Это его вектор оценок: отсутствующие оценки считаем нулевыми.


In [32]:
def is_rating(row):
    return row[0] != "userId"


def parse_rating(row):
    return int(row[1]), (int(row[0]), float(row[2]))


ratings = (
    sc.textFile("ml-latest-small/ratings.csv")
    .mapPartitions(csv.reader)
    .filter(is_rating)
    .map(parse_rating)
)
movie_vectors = ratings.groupByKey().mapValues(dict).cache()
print(f"Фильмов с оценками: {movie_vectors.count()}")


Фильмов с оценками: 9724


## 1. Косинусное сходство

Функция принимает два словаря оценок и возвращает одно число. Сама вычисляет скалярное произведение и длины векторов.

$$\operatorname{sim}(i,j)=\frac{\sum_u r_{ui}r_{uj}}{\sqrt{\sum_u r_{ui}^2}\sqrt{\sum_u r_{uj}^2}}.$$



In [33]:
def cosine_similarity(first, second):
    dot_product = sum(rating * second.get(user, 0) for user, rating in first.items())
    norm1 = sqrt(sum(rating ** 2 for rating in first.values()))
    norm2 = sqrt(sum(rating ** 2 for rating in second.values()))
    return dot_product / (norm1 * norm2)


first_movie, second_movie = movie_vectors.take(2)
print(f"Сходство фильмов {first_movie[0]} и {second_movie[0]}: "
      f"{cosine_similarity(first_movie[1], second_movie[1]):.6f}")


Сходство фильмов 6 и 50: 0.456850


## 2. RDD сходств с фильмом 589

Получаем вектор фильма `589` и сравниваем его с каждым другим фильмом. Элемент результата — `(movieId, similarity)`.


In [37]:
target_movie = 589


def is_target_movie(row):
    return row[0] == target_movie


def is_other_movie(row):
    return row[0] != target_movie


def compare_with_target(row):
    movie_id, vector = row
    return movie_id, cosine_similarity(target_vector, vector)


target_vector = movie_vectors.filter(is_target_movie).values().first()
similarities = movie_vectors.filter(is_other_movie).map(compare_with_target).cache()
print(f"Сходств с фильмом {target_movie}: {similarities.count()}")


Сходств с фильмом 589: 9723


## 3. Названия фильмов

Добавляем названия к RDD по `movieId`.


In [38]:
def is_movie(row):
    return row[0] != "movieId"


def parse_movie(row):
    return int(row[0]), row[1]


def add_title(row):
    movie_id, (similarity, title) = row
    return {"movieId": movie_id, "title": title, "similarity": similarity}


titles = (
    sc.textFile("ml-latest-small/movies.csv")
    .mapPartitions(csv.reader)
    .filter(is_movie)
    .map(parse_movie)
)
movies_with_similarity = similarities.join(titles).map(add_title)
movies_with_similarity.take(3)

[{'movieId': 216,
  'title': 'Billy Madison (1995)',
  'similarity': 0.23031120299047797},
 {'movieId': 260,
  'title': 'Star Wars: Episode IV - A New Hope (1977)',
  'similarity': 0.5900777578543723},
 {'movieId': 296,
  'title': 'Pulp Fiction (1994)',
  'similarity': 0.6102841093789623}]

## 4. Топ-10 похожих фильмов


In [39]:
def by_similarity(movie):
    return movie["similarity"]


def show_movies(rows):
    print(f"{'movieId':<8} {'Сходство':>10} Название")
    for movie in rows:
        print(f"{movie['movieId']:<8} {movie['similarity']:>10.6f} {movie['title']}")


top_movies = movies_with_similarity.top(10, key=by_similarity)
show_movies(top_movies)


movieId    Сходство Название
480        0.719983 Jurassic Park (1993)
1240       0.695724 Terminator, The (1984)
110        0.659827 Braveheart (1995)
592        0.645603 Batman (1989)
457        0.637561 Fugitive, The (1993)
377        0.630092 Speed (1994)
1196       0.618530 Star Wars: Episode V - The Empire Strikes Back (1980)
380        0.611164 True Lies (1994)
296        0.610284 Pulp Fiction (1994)
356        0.600886 Forrest Gump (1994)
